In [90]:
!pkill -f flask
!pkill -f ngrok

In [91]:
!pip install flask flask-ngrok pyngrok

In [92]:
!pip install fpdf

In [93]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [94]:
!ls "/content/drive/MyDrive/DRD_UI_4225"

'efficientnetb5_trained (4).pth'


In [95]:
!pip install pytz -q

In [96]:
%%writefile app.py

from flask import Flask, render_template, request, redirect, session, send_file
import torch
import timm
import sqlite3
import cv2
import numpy as np
from PIL import Image
from fpdf import FPDF
import os
import uuid
from datetime import datetime
import pytz

app = Flask(__name__)
app.secret_key = "dr_secret_key"

UPLOAD_FOLDER = "static/uploads"
REPORT_FOLDER = "reports"

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(REPORT_FOLDER, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# LOAD MODEL
# ================================
model = timm.create_model('tf_efficientnet_b5', pretrained=False, num_classes=5)
model.load_state_dict(
    torch.load("/content/drive/MyDrive/DRD_UI_4225/efficientnetb5_trained (4).pth",
               map_location=device)
)
model.to(device)
model.eval()

class_names = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]

stage_meta = {
    "No DR":            {"stage": 0, "color": "#00e5a0", "rec": "Routine annual screening recommended. Maintain blood glucose and BP control."},
    "Mild DR":          {"stage": 1, "color": "#ffe064", "rec": "Follow-up in 12 months. Optimize glycemic control and manage systemic risk factors."},
    "Moderate DR":      {"stage": 2, "color": "#ffb347", "rec": "Refer to ophthalmologist within 6 months. Tighten glycemic control urgently."},
    "Severe DR":        {"stage": 3, "color": "#ff7832", "rec": "Urgent ophthalmology referral within 1 month. Panretinal photocoagulation may be needed."},
    "Proliferative DR": {"stage": 4, "color": "#ff4c6a", "rec": "WARNING: Immediate referral. High risk of vitreous hemorrhage & blindness. Laser/injection therapy needed."},
}


# ================================
# IMAGE PREPROCESSING
# ================================
def crop_retina(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) != 0:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        img = img[y + h // 4: y + h, x: x + w]
    return img


def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


def preprocess(img):
    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img = crop_retina(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (456, 456))
    img = img / 255.0
    img = torch.tensor(img).permute(2, 0, 1).float().unsqueeze(0)
    return img.to(device)


# ================================
# DATABASE
# ================================
def get_db():
    return sqlite3.connect("database.db")


def init_db():
    conn = sqlite3.connect("database.db")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            username  TEXT PRIMARY KEY,
            password  TEXT,
            full_name TEXT,
            hospital  TEXT
        )
    """)
    conn.commit()
    conn.close()


init_db()


# ================================
# SAFE TEXT FOR FPDF (latin-1 only)
# ================================
def safe(text):
    return (str(text)
            .replace('\u2014', '-')
            .replace('\u2013', '-')
            .replace('\u2019', "'")
            .replace('\u2018', "'")
            .replace('\u201c', '"')
            .replace('\u201d', '"')
            .encode('latin-1', 'replace')
            .decode('latin-1'))


# ================================
# HOME
# ================================
@app.route("/")
def home():
    if "user" in session:
        return redirect("/dashboard")
    return redirect("/login")


# ================================
# REGISTER
# ================================
@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username  = request.form["username"]
        password  = request.form["password"]
        full_name = request.form.get("full_name", "")
        hospital  = request.form.get("hospital", "")

        conn = get_db()
        cur  = conn.cursor()
        cur.execute("SELECT * FROM users WHERE username=?", (username,))
        if cur.fetchone():
            conn.close()
            return render_template("register.html", error="Username already exists.")

        cur.execute("INSERT INTO users VALUES (?,?,?,?)",
                    (username, password, full_name, hospital))
        conn.commit()
        conn.close()
        return redirect("/login")

    return render_template("register.html")


# ================================
# LOGIN
# ================================
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        conn = get_db()
        cur  = conn.cursor()
        cur.execute("SELECT * FROM users WHERE username=? AND password=?",
                    (username, password))
        data = cur.fetchone()
        conn.close()

        if data:
            session["user"]      = data[0]
            session["full_name"] = data[2] or data[0]
            session["hospital"]  = data[3] or "-"
            return redirect("/dashboard")
        else:
            return render_template("login.html", error="Invalid username or password.")

    return render_template("login.html")


# ================================
# DASHBOARD
# ================================
@app.route("/dashboard")
def dashboard():
    if "user" not in session:
        return redirect("/login")
    return render_template("dashboard.html",
                           user=session.get("full_name", session["user"]))


# ================================
# PREDICT
# ================================
@app.route("/predict", methods=["POST"])
def predict():
    if "user" not in session:
        return redirect("/login")

    # ── Save & convert upload to real JPEG ─────────────
    file     = request.files["file"]
    filename = str(uuid.uuid4()) + ".jpg"
    path     = os.path.join(UPLOAD_FOLDER, filename)
    file.save(path)

    # Force re-encode as proper JPEG (fixes FPDF crash on PNG/WebP)
    pil_img = Image.open(path).convert("RGB")
    pil_img.save(path, "JPEG", quality=95)

    # ── Run model ───────────────────────────────────────
    tensor = preprocess(pil_img)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, 1)[0].tolist()
        conf, pred = torch.max(torch.tensor(probs), 0)

    prediction = class_names[pred.item()]
    confidence = round(conf.item() * 100, 2)
    meta       = stage_meta[prediction]

    # ── Patient details ─────────────────────────────────
    patient = {
        "name":        request.form.get("pt_name",     "-"),
        "pid":         request.form.get("pt_id",       "-"),
        "age":         request.form.get("pt_age",      "-"),
        "gender":      request.form.get("pt_gender",   "-"),
        "city":       request.form.get("pt_city",    "-"),
        "eye":    request.form.get("pt_eye", "-"),
        "doctor":      request.form.get("pt_doctor",   "-"),
        "notes":       request.form.get("pt_notes",    "-"),
        "hospital":    session.get("hospital",   "-"),
        "analyzed_by": session.get("full_name",  session["user"]),
        "date": datetime.now(pytz.timezone("Asia/Kolkata")).strftime("%d %b %Y, %H:%M"),
    }

    # ── Probability list ────────────────────────────────
    prob_list = [
        {"name": class_names[i], "pct": round(probs[i] * 100, 2),
         "color": stage_meta[class_names[i]]["color"]}
        for i in range(5)
    ]

    # ── Generate PDF ────────────────────────────────────
    report_path = generate_report(prediction, confidence, patient, prob_list, path)

    return render_template(
        "result.html",
        prediction=prediction,
        confidence=confidence,
        stage=meta["stage"],
        stage_color=meta["color"],
        recommendation=meta["rec"],
        report=report_path,
        image=path,
        patient=patient,
        prob_list=prob_list,
    )


# ================================
# PDF REPORT
# ================================
def generate_report(prediction, confidence, patient, prob_list, img_path):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # ── Header ──────────────────────────────────────────
    pdf.set_fill_color(7, 21, 37)
    pdf.rect(0, 0, 210, 40, "F")
    pdf.set_text_color(0, 198, 255)
    pdf.set_font("Arial", "B", 20)
    pdf.set_xy(10, 10)
    pdf.cell(0, 10, "RetinaAI - Diabetic Retinopathy Report", ln=True)
    pdf.set_font("Arial", "", 9)
    pdf.set_text_color(90, 122, 154)
    pdf.cell(0, 8, safe(f"Generated: {patient['date']}   |   Model: EfficientNet-B5   |   Dataset: APTOS 2019"), ln=True)

    pdf.ln(10)

    # ── Patient Information ──────────────────────────────
    pdf.set_font("Arial", "B", 13)
    pdf.set_text_color(0, 114, 255)
    pdf.cell(0, 8, "Patient Information", ln=True)
    pdf.set_draw_color(0, 114, 255)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(4)

    rows = [
        ("Patient Name",     patient["name"]),
        ("Patient ID",       patient["pid"]),
        ("Age",              patient["age"]),
        ("Gender",           patient["gender"]),
        ("City",             patient["city"]),
        ("Eye",              patient["eye"]),
        ("Referring Doctor", patient["doctor"]),
        ("Institution",      patient["hospital"]),
        ("Analyzed By",      patient["analyzed_by"]),
    ]
    for label, value in rows:
        pdf.set_font("Arial", "B", 10)
        pdf.set_text_color(30, 30, 30)
        pdf.cell(60, 7, safe(label) + ":", border=0)
        pdf.set_font("Arial", "", 10)
        pdf.cell(0, 7, safe(value), ln=True)

    if patient["notes"] not in ("-", "", None):
        pdf.set_font("Arial", "B", 10)
        pdf.set_text_color(30, 30, 30)
        pdf.cell(60, 7, "Clinical Notes:", border=0)
        pdf.set_font("Arial", "", 10)
        pdf.multi_cell(0, 7, safe(patient["notes"]))

    pdf.ln(6)

    # ── Diagnosis Result ─────────────────────────────────
    pdf.set_font("Arial", "B", 13)
    pdf.set_text_color(0, 114, 255)
    pdf.cell(0, 8, "AI Diagnosis Result", ln=True)
    pdf.set_draw_color(0, 114, 255)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(4)

    pdf.set_font("Arial", "B", 14)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 10, safe(f"Stage {prediction}   |   Confidence: {confidence}%"), ln=True)

    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(80, 80, 80)
    rec = stage_meta[prediction]["rec"].replace("WARNING: ", "")
    pdf.multi_cell(0, 7, safe(f"Recommendation: {rec}"))
    pdf.ln(4)

    # ── Probability Table ────────────────────────────────
    pdf.set_font("Arial", "B", 11)
    pdf.set_text_color(0, 114, 255)
    pdf.cell(0, 8, "Stage Probability Distribution", ln=True)
    pdf.ln(2)

    pdf.set_font("Arial", "B", 9)
    pdf.set_fill_color(230, 240, 255)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(80, 7, "Stage", border=1, fill=True)
    pdf.cell(40, 7, "Probability", border=1, fill=True, ln=True)

    pdf.set_font("Arial", "", 9)
    for p in prob_list:
        pdf.cell(80, 6, safe(p["name"]), border=1)
        pdf.cell(40, 6, f"{p['pct']}%", border=1, ln=True)

    pdf.ln(6)

    # ── Fundus Image ─────────────────────────────────────
    if os.path.exists(img_path):
        try:
            safe_path = img_path + "_embed.jpg"
            Image.open(img_path).convert("RGB").save(safe_path, "JPEG", quality=90)
            pdf.set_font("Arial", "B", 11)
            pdf.set_text_color(0, 114, 255)
            pdf.cell(0, 8, "Uploaded Fundus Image", ln=True)
            pdf.ln(2)
            pdf.image(safe_path, x=10, w=80)
            pdf.ln(4)
            os.remove(safe_path)
        except Exception as img_err:
            pdf.set_font("Arial", "I", 9)
            pdf.set_text_color(150, 0, 0)
            pdf.cell(0, 8, safe(f"[Image unavailable: {img_err}]"), ln=True)

    # ── Disclaimer ────────────────────────────────────────
    pdf.set_font("Arial", "I", 8)
    pdf.set_text_color(120, 120, 120)
    pdf.multi_cell(0, 5, safe(
        "DISCLAIMER: This report is generated by an AI-assisted screening tool and is intended to "
        "support, not replace, clinical diagnosis. Final interpretation must be made by a qualified "
        "ophthalmologist or retinal specialist."
    ))

    filename = f"{REPORT_FOLDER}/report_{uuid.uuid4()}.pdf"
    pdf.output(filename)
    return filename


# ================================
# DOWNLOAD
# ================================
@app.route("/download/<path:path>")
def download(path):
    return send_file(path, as_attachment=True)


# ================================
# LOGOUT
# ================================
@app.route("/logout")
def logout():
    session.clear()
    return redirect("/login")


# ================================
# RUN
# ================================
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000, debug=False)

Overwriting app.py


In [97]:
!mkdir templates
!mkdir static

mkdir: cannot create directory ‘templates’: File exists
mkdir: cannot create directory ‘static’: File exists


In [98]:
%%writefile templates/login.html

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RetinaAI — Login</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<!-- BG LAYERS -->
<div class="bg-mesh"></div>
<div class="grid-lines"></div>
<div class="retina-ring"></div>

<!-- NAVBAR -->
<nav>
  <div class="logo">
    <div class="logo-eye">👁</div>
    Retina<span style="-webkit-text-fill-color:var(--text)">AI</span>
  </div>
  <div class="nav-right">
    <a href="/register" class="nav-btn primary">Create Account</a>
  </div>
</nav>

<!-- LOGIN CARD -->
<div class="page-center">
  <div class="glass-card sm">

    <h2 class="fade-up">Welcome Back</h2>
    <p class="subtitle fade-up-1">Sign in to access the DR detection system.</p>

    {% if error %}
    <div class="alert error">{{ error }}</div>
    {% endif %}

    <form method="POST" class="fade-up-2">

      <div class="field">
        <label>Username / Doctor ID</label>
        <input type="text" name="username" placeholder="Enter your username" required autocomplete="username">
      </div>

      <div class="field">
        <label>Password</label>
        <div class="input-wrap">
          <input type="password" name="password" id="pass" placeholder="Your password" required autocomplete="current-password">
          <button type="button" class="eye-btn" onclick="togglePwd('pass',this)">👁</button>
        </div>
      </div>

      <button type="submit" class="btn btn-primary btn-full btn-lg" style="margin-top:8px;">
        Sign In →
      </button>
    </form>

    <p style="text-align:center; margin-top:22px; font-size:.88rem; color:var(--muted);" class="fade-up-3">
      No account? <a href="/register" style="color:var(--accent); text-decoration:none;">Create one free</a>
    </p>

  </div>
</div>

<!-- TOAST (unused on this page but included for consistency) -->
<div class="toast" id="toast"></div>

<script>
function togglePwd(id, btn) {
  const input = document.getElementById(id);
  input.type = input.type === 'password' ? 'text' : 'password';
  btn.textContent = input.type === 'password' ? '👁' : '🙈';
}
</script>
</body>
</html>

Overwriting templates/login.html


In [99]:
%%writefile templates/register.html

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RetinaAI — Create Account</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<div class="bg-mesh"></div>
<div class="grid-lines"></div>
<div class="retina-ring"></div>

<!-- NAVBAR -->
<nav>
  <div class="logo">
    <div class="logo-eye">👁</div>
    Retina<span style="-webkit-text-fill-color:var(--text)">AI</span>
  </div>
  <div class="nav-right">
    <a href="/login" class="nav-btn ghost">Sign In</a>
  </div>
</nav>

<!-- REGISTER CARD -->
<div class="page-center">
  <div class="glass-card sm" style="max-width:500px;">

    <h2 class="fade-up">Create Account</h2>
    <p class="subtitle fade-up-1">Join RetinaAI and start detecting diabetic retinopathy.</p>

    {% if error %}
    <div class="alert error">{{ error }}</div>
    {% endif %}

    <form method="POST" class="fade-up-2">

      <!-- Full Name + Username side by side -->
      <div class="form-row">
        <div class="field">
          <label>Full Name</label>
          <input type="text" name="full_name" placeholder="Dr. John Doe">
        </div>
        <div class="field">
          <label>Username *</label>
          <input type="text" name="username" placeholder="Unique ID" required autocomplete="username">
        </div>
      </div>

      <!-- Hospital -->
      <div class="field">
        <label>Hospital / Institution</label>
        <input type="text" name="hospital" placeholder="City Medical Center">
      </div>

      <!-- Password -->
      <div class="field">
        <label>Password *</label>
        <div class="input-wrap">
          <input type="password" name="password" id="pass1" placeholder="Min. 8 characters" required autocomplete="new-password">
          <button type="button" class="eye-btn" onclick="togglePwd('pass1',this)">👁</button>
        </div>
      </div>

      <!-- Confirm Password (client-side only check) -->
      <div class="field">
        <label>Confirm Password *</label>
        <div class="input-wrap">
          <input type="password" id="pass2" placeholder="Repeat password">
          <button type="button" class="eye-btn" onclick="togglePwd('pass2',this)">👁</button>
        </div>
      </div>

      <div class="alert error" id="pw-mismatch" style="display:none;">Passwords do not match.</div>

      <button type="submit" class="btn btn-primary btn-full btn-lg" style="margin-top:8px;" onclick="return checkPwd()">
        Create Account →
      </button>
    </form>

    <p style="text-align:center; margin-top:22px; font-size:.88rem; color:var(--muted);" class="fade-up-3">
      Already have an account? <a href="/login" style="color:var(--accent); text-decoration:none;">Sign in</a>
    </p>

  </div>
</div>

<div class="toast" id="toast"></div>

<script>
function togglePwd(id, btn) {
  const input = document.getElementById(id);
  input.type = input.type === 'password' ? 'text' : 'password';
  btn.textContent = input.type === 'password' ? '👁' : '🙈';
}
function checkPwd() {
  const p1 = document.getElementById('pass1').value;
  const p2 = document.getElementById('pass2').value;
  if (p2 && p1 !== p2) {
    document.getElementById('pw-mismatch').style.display = 'block';
    return false;
  }
  return true;
}
</script>
</body>
</html>


Overwriting templates/register.html


In [100]:
%%writefile templates/dashboard.html

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RetinaAI — Dashboard</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<!-- LOADING OVERLAY (shown during form submit) -->
<div class="loader-overlay" id="loader">
  <div class="loader-ring"></div>
  <div class="loader-text">Running EfficientNet-B5 analysis…</div>
</div>

<div class="bg-mesh"></div>
<div class="grid-lines"></div>

<!-- NAVBAR -->
<nav>
  <div class="logo">
    <div class="logo-eye">👁</div>
    Retina<span style="-webkit-text-fill-color:var(--text)">AI</span>
  </div>
  <div class="nav-right">
    <div class="nav-user">
      <div class="user-avatar">{{ user[0]|upper }}</div>
      <span style="display:none;" id="uname">{{ user }}</span>
    </div>
    <a href="/logout" class="nav-btn ghost">Logout</a>
  </div>
</nav>

<!-- MAIN CONTENT -->
<div class="page-body">

  <!-- Page header -->
  <div class="page-header fade-up">
    <div>
      <h2>New Diagnosis</h2>
      <p class="meta">Welcome, {{ user }}. Upload a retinal fundus image to begin AI analysis.</p>
    </div>
  </div>

  <!-- FORM wraps upload zone + patient details -->
  <form action="/predict" method="POST" enctype="multipart/form-data" id="analysis-form">

    <!-- ── UPLOAD ZONE ── -->
    <div class="upload-zone fade-up-1" id="upload-zone">
      <input type="file" name="file" id="file-input" accept="image/*" required onchange="handleFile(event)">
      <div id="upload-placeholder">
        <span class="upload-icon">🔬</span>
        <h3>Drop Retinal Fundus Image Here</h3>
        <p>PNG / JPG / JPEG · Max 10 MB · Fundus photography required</p>
      </div>
      <img id="preview-img" class="preview-img" src="" alt="Preview">
    </div>

    <!-- ── PATIENT INFORMATION ── -->
    <div class="section-card fade-up-2">
      <div class="section-title">
        <div class="icon-badge">👤</div>
        Patient Information
      </div>

      <div class="form-grid">

        <div class="field">
          <label>Patient Name *</label>
          <input type="text" name="pt_name" placeholder="Full name" required>
        </div>

        <div class="field">
          <label>Patient ID</label>
          <input type="text" name="pt_id" placeholder="e.g. PT-2024-001">
        </div>

        <div class="field">
          <label>Age</label>
          <input type="number" name="pt_age" placeholder="Years" min="1" max="120">
        </div>

        <div class="field">
          <label>Gender</label>
          <select name="pt_gender">
            <option value="-">Select</option>
            <option value="Male">Male</option>
            <option value="Female">Female</option>
            <option value="Other">Other</option>
          </select>
        </div>

        <div class="field">
          <label>City</label>
          <input type="text" name="pt_city" placeholder="e.g. Hyderabad">
        </div>

        <div class="field">
          <label>Which Eye</label>
          <select name="pt_eye">
            <option value="-">Select</option>
            <option value="Right Eye">Right Eye</option>
            <option value="Left Eye">Left Eye</option>
            <option value="Both Eyes">Both Eyes</option>
          </select>
        </div>

        <div class="field full">
          <label>Referring Doctor</label>
          <input type="text" name="pt_doctor" placeholder="Dr. Full Name">
        </div>

        <div class="field full">
          <label>Clinical Notes (optional)</label>
          <input type="text" name="pt_notes" placeholder="Any relevant observations…">
        </div>

      </div>
    </div>

    <!-- ── SUBMIT ── -->
    <button type="submit" class="analyze-btn fade-up-3" id="analyze-btn">
      <svg width="20" height="20" fill="none" viewBox="0 0 24 24" stroke="currentColor" stroke-width="2">
        <circle cx="11" cy="11" r="8"/><path stroke-linecap="round" stroke-linejoin="round" d="M21 21l-4.35-4.35"/>
      </svg>
      Run AI Analysis
    </button>

  </form>

</div><!-- /page-body -->

<div class="toast" id="toast"></div>

<script>
// ── File preview ───────────────────────────────────────
function handleFile(e) {
  const file = e.target.files[0];
  if (!file) return;
  if (!file.type.startsWith('image/')) { showToast('Please upload an image file.', 'error'); return; }
  if (file.size > 10 * 1024 * 1024)   { showToast('File too large. Max 10 MB.', 'error');   return; }

  const reader = new FileReader();
  reader.onload = ev => {
    const img = document.getElementById('preview-img');
    img.src = ev.target.result;
    img.style.display = 'block';
  };
  reader.readAsDataURL(file);

  document.getElementById('upload-placeholder').innerHTML =
    `<span class="upload-icon">✅</span>
     <h3>${file.name}</h3>
     <p>${(file.size/1024).toFixed(0)} KB · Click to change</p>`;
}

// ── Drag & drop ────────────────────────────────────────
const zone = document.getElementById('upload-zone');
zone.addEventListener('dragover',  e => { e.preventDefault(); zone.classList.add('drag-over'); });
zone.addEventListener('dragleave', ()  => zone.classList.remove('drag-over'));
zone.addEventListener('drop', e => {
  e.preventDefault(); zone.classList.remove('drag-over');
  const dt = e.dataTransfer;
  if (dt.files[0]) {
    document.getElementById('file-input').files = dt.files;
    handleFile({ target: { files: dt.files } });
  }
});

// ── Show loader on submit ──────────────────────────────
document.getElementById('analysis-form').addEventListener('submit', function(e) {
  const file = document.getElementById('file-input').files[0];
  if (!file) { e.preventDefault(); showToast('Please upload a retinal image first.', 'error'); return; }
  document.getElementById('loader').classList.add('show');
  document.getElementById('analyze-btn').disabled = true;
});

// ── Toast helper ───────────────────────────────────────
let toastTimer;
function showToast(msg, type = 'success') {
  const t = document.getElementById('toast');
  t.textContent = (type === 'success' ? '✓ ' : '✕ ') + msg;
  t.className = 'toast show ' + type;
  clearTimeout(toastTimer);
  toastTimer = setTimeout(() => t.classList.remove('show'), 3200);
}
</script>
</body>
</html>


Overwriting templates/dashboard.html


In [101]:
%%writefile templates/result.html

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RetinaAI — Analysis Report</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<div class="bg-mesh"></div>
<div class="grid-lines"></div>

<!-- NAVBAR -->
<nav>
  <div class="logo">
    <div class="logo-eye">👁</div>
    Retina<span style="-webkit-text-fill-color:var(--text)">AI</span>
  </div>
  <div class="nav-right">
    <a href="/dashboard" class="nav-btn ghost">← New Analysis</a>
    <a href="/logout"    class="nav-btn ghost">Logout</a>
  </div>
</nav>

<!-- MAIN CONTENT -->
<div class="page-body wide">

  <!-- Page header -->
  <div class="page-header fade-up">
    <div>
      <h2>Analysis Report</h2>
      <p class="meta">
        Completed: {{ patient.date }} &nbsp;·&nbsp;
        Model: EfficientNet-B5 &nbsp;·&nbsp;
        Analyzed by: {{ patient.analyzed_by }}
      </p>
    </div>
  </div>

  <!-- Two-column grid -->
  <div class="result-grid">

    <!-- ════ LEFT: image + patient table ════ -->
    <div>

      <!-- Fundus image -->
      <div class="img-card fade-up-1">
        <img src="{{ image }}" alt="Retinal fundus image">
        <div class="img-meta">
          <h4>Retinal Fundus Image</h4>
          <p>{{ patient.name }} &nbsp;·&nbsp; {{ patient.date }}</p>
        </div>
      </div>

      <!-- Patient details -->
      <div class="patient-card fade-up-2">
        <div class="section-title">
          <div class="icon-badge">📋</div>
          Patient Details
        </div>
        <table class="info-table">
          <tr><td>Patient Name</td>   <td>{{ patient.name }}</td></tr>
          <tr><td>Patient ID</td>     <td>{{ patient.pid }}</td></tr>
          <tr><td>Age</td>            <td>{{ patient.age }}</td></tr>
          <tr><td>Gender</td>         <td>{{ patient.gender }}</td></tr>
          <tr><td>City</td>  <td>{{ patient.city }}</td></tr>
          <tr><td>Eye</td>
              <td>{{ patient.eye }}</td>
          </tr>
          <tr><td>Referring Doctor</td><td>{{ patient.doctor }}</td></tr>
          <tr><td>Institution</td>    <td>{{ patient.hospital }}</td></tr>
          <tr><td>Analyzed By</td>    <td>{{ patient.analyzed_by }}</td></tr>
          {% if patient.notes != '—' %}
          <tr><td>Clinical Notes</td> <td>{{ patient.notes }}</td></tr>
          {% endif %}
        </table>
      </div>

    </div><!-- /left -->

    <!-- ════ RIGHT: diagnosis results ════ -->
    <div class="result-card fade-up-1">

      <p class="result-label">DR Classification Result</p>

      <!-- Stage badge — class stage-0 … stage-4 -->
      <div class="dr-stage stage-{{ stage }}">
        Stage {{ stage }} — {{ prediction }}
      </div>

      <!-- Short description per stage -->
      <p class="stage-desc">
        {% if stage == 0 %}No signs of diabetic retinopathy detected. Retina appears normal.
        {% elif stage == 1 %}Mild Non-Proliferative DR. Early microaneurysms present.
        {% elif stage == 2 %}Moderate Non-Proliferative DR. Multiple microaneurysms and hemorrhages visible.
        {% elif stage == 3 %}Severe Non-Proliferative DR. Extensive retinal hemorrhages detected.
        {% elif stage == 4 %}⚠ Proliferative DR. Neovascularisation detected — immediate care required.
        {% endif %}
      </p>

      <!-- Confidence bar -->
      <div class="conf-section">
        <div class="conf-row">
          <span>Confidence Score</span>
          <span class="conf-pct">{{ confidence }}%</span>
        </div>
        <div class="bar-track">
          <div class="bar-fill" id="conf-bar" style="width: 0%"></div>
        </div>
      </div>

      <!-- All stage probabilities -->
      <div class="probs-section">
        <div class="probs-label">Stage Probability Distribution</div>
        {% for p in prob_list %}
        <div class="prob-row">
          <div class="prob-name">{{ p.name }}</div>
          <div class="prob-mini-track">
            <div class="prob-mini-fill"
                 id="prob-{{ loop.index0 }}"
                 style="width:0%; background:{{ p.color }}">
            </div>
          </div>
          <div class="prob-pct" style="color:{{ p.color }}">{{ p.pct }}%</div>
        </div>
        {% endfor %}
      </div>

      <!-- Clinical recommendation -->
      <div class="rec-box rec-{{ stage }}">
        {{ recommendation }}
      </div>

      <!-- Action buttons -->
      <div class="result-actions">
        <a href="/download/{{ report }}" class="btn btn-success">
          ⬇ Download PDF Report
        </a>
        <a href="/dashboard" class="btn btn-outline">
          New Scan
        </a>
      </div>

    </div><!-- /result-card -->
  </div><!-- /result-grid -->

</div><!-- /page-body -->

<div class="toast" id="toast"></div>

<script>
// Prob data injected from Flask
const probData = [
  {% for p in prob_list %}{ pct: {{ p.pct }} }{% if not loop.last %},{% endif %}{% endfor %}
];
const confVal = {{ confidence }};

// Animate bars after page load
window.addEventListener('load', () => {
  setTimeout(() => {
    document.getElementById('conf-bar').style.width = confVal + '%';
    probData.forEach((p, i) => {
      const el = document.getElementById('prob-' + i);
      if (el) el.style.width = p.pct + '%';
    });
  }, 120);
});

// Toast helper
let toastTimer;
function showToast(msg, type = 'success') {
  const t = document.getElementById('toast');
  t.textContent = (type === 'success' ? '✓ ' : '✕ ') + msg;
  t.className = 'toast show ' + type;
  clearTimeout(toastTimer);
  toastTimer = setTimeout(() => t.classList.remove('show'), 3200);
}
</script>
</body>
</html>


Overwriting templates/result.html


In [102]:
%%writefile static/style.css

/* =============================================
   RetinaAI — Global Stylesheet
   Used by: login, register, dashboard, result
   ============================================= */

@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:ital,wght@0,300;0,400;0,500;1,300&display=swap');

/* ── VARIABLES ── */
:root {
  --bg:       #00070a;
  --surface:  #071525;
  --surface2: #0c2035;
  --border:   rgba(0,180,255,0.13);
  --accent:   #00c6ff;
  --accent2:  #0072ff;
  --danger:   #ff4c6a;
  --warn:     #ffb347;
  --success:  #00e5a0;
  --text:     #e8f4ff;
  --muted:    #5a7a9a;
  --glow:     0 0 40px rgba(0,198,255,0.15);
}

/* ── RESET ── */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
html { scroll-behavior: smooth; }

body {
  font-family: 'DM Sans', sans-serif;
  background: var(--bg);
  color: var(--text);
  min-height: 100vh;
  overflow-x: hidden;
}

/* ── BACKGROUND LAYERS ── */
.bg-mesh {
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background:
    radial-gradient(ellipse 80% 60% at 15% -5%,  rgba(0,114,255,0.13) 0%, transparent 60%),
    radial-gradient(ellipse 60% 50% at 90% 100%, rgba(0,198,255,0.08) 0%, transparent 60%),
    radial-gradient(ellipse 40% 40% at 50% 50%,  rgba(0,30,60,0.7)    0%, transparent 100%);
}
.grid-lines {
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background-image:
    linear-gradient(rgba(0,198,255,0.035) 1px, transparent 1px),
    linear-gradient(90deg, rgba(0,198,255,0.035) 1px, transparent 1px);
  background-size: 64px 64px;
}
.retina-ring {
  position: fixed;
  width: 720px; height: 720px;
  border-radius: 50%;
  border: 1px solid rgba(0,198,255,0.05);
  top: 50%; left: 50%;
  transform: translate(-50%, -50%);
  z-index: 0; pointer-events: none;
  animation: slowspin 50s linear infinite;
}
.retina-ring::before {
  content: '';
  position: absolute; inset: 60px;
  border-radius: 50%;
  border: 1px solid rgba(0,198,255,0.04);
}
@keyframes slowspin { to { transform: translate(-50%,-50%) rotate(360deg); } }

/* ── NAVBAR ── */
nav {
  position: sticky; top: 0; z-index: 100;
  display: flex; align-items: center; justify-content: space-between;
  padding: 18px 48px;
  background: rgba(3,11,20,0.85);
  backdrop-filter: blur(20px);
  border-bottom: 1px solid var(--border);
}
.logo {
  display: flex; align-items: center; gap: 10px;
  font-family: 'Syne', sans-serif; font-weight: 800; font-size: 1.35rem;
  background: linear-gradient(135deg, var(--accent), var(--accent2));
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  letter-spacing: -0.5px;
}
.logo-eye {
  width: 34px; height: 34px; border-radius: 50%; flex-shrink: 0;
  background: linear-gradient(135deg, var(--accent2), var(--accent));
  display: flex; align-items: center; justify-content: center;
  font-size: .95rem; box-shadow: 0 0 18px rgba(0,198,255,0.4);
  animation: eyePulse 3s ease-in-out infinite;
  -webkit-text-fill-color: initial;
}
@keyframes eyePulse {
  0%,100% { box-shadow: 0 0 18px rgba(0,198,255,0.4); }
  50%      { box-shadow: 0 0 32px rgba(0,198,255,0.75); }
}
.nav-right { display: flex; align-items: center; gap: 14px; }
.nav-user {
  display: flex; align-items: center; gap: 10px;
  font-size: .88rem; color: var(--muted);
}
.user-avatar {
  width: 36px; height: 36px; border-radius: 50%; flex-shrink: 0;
  background: linear-gradient(135deg, var(--accent2), var(--accent));
  display: flex; align-items: center; justify-content: center;
  font-family: 'Syne', sans-serif; font-weight: 700; font-size: .95rem; color: #fff;
}
.nav-btn {
  padding: 8px 20px; border-radius: 8px; border: none; cursor: pointer;
  font-family: 'DM Sans', sans-serif; font-size: .86rem; font-weight: 500;
  transition: all .2s;
}
.nav-btn.ghost {
  background: transparent; color: var(--muted); border: 1px solid var(--border);
}
.nav-btn.ghost:hover { color: var(--danger); border-color: var(--danger); }
.nav-btn.primary {
  background: linear-gradient(135deg, var(--accent2), var(--accent)); color: #fff;
}
.nav-btn.primary:hover { transform: translateY(-1px); box-shadow: 0 6px 20px rgba(0,114,255,0.35); }

/* ── LAYOUT HELPERS ── */
.page-center {
  position: relative; z-index: 1;
  display: flex; align-items: center; justify-content: center;
  min-height: calc(100vh - 70px); padding: 40px 24px;
}
.page-body {
  position: relative; z-index: 1;
  max-width: 960px; margin: 0 auto;
  padding: 52px 24px 60px;
}
.page-body.wide { max-width: 1160px; }

/* ── GLASS CARD ── */
.glass-card {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: 20px;
  padding: 44px 40px;
  box-shadow: var(--glow);
  animation: fadeUp .5s ease both;
}
.glass-card.sm { max-width: 460px; width: 100%; }
.glass-card h2 {
  font-family: 'Syne', sans-serif; font-weight: 700; font-size: 1.85rem; margin-bottom: 6px;
}
.glass-card .subtitle { color: var(--muted); font-size: .9rem; margin-bottom: 32px; }

/* ── ALERT BANNERS ── */
.alert {
  padding: 10px 14px; border-radius: 9px;
  font-size: .84rem; margin-bottom: 18px; display: none;
}
.alert.error  { background: rgba(255,76,106,0.1);  border: 1px solid rgba(255,76,106,0.3);  color: #ff8aa2; display: block; }
.alert.success{ background: rgba(0,229,160,0.1);   border: 1px solid rgba(0,229,160,0.3);   color: var(--success); display: block; }

/* ── FORM ELEMENTS ── */
.form-row { display: flex; gap: 16px; }
.form-row .field { flex: 1; }
.form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.form-grid .field { margin-bottom: 0; }
.form-grid .field.full { grid-column: 1 / -1; }

.field { margin-bottom: 18px; }
.field label {
  display: block; font-size: .79rem; font-weight: 500;
  color: var(--muted); margin-bottom: 7px; letter-spacing: .3px; text-transform: uppercase;
}
.field input, .field select {
  width: 100%; padding: 12px 15px;
  background: var(--surface2); border: 1px solid var(--border);
  border-radius: 10px; color: var(--text);
  font-family: 'DM Sans', sans-serif; font-size: .93rem;
  transition: border-color .2s, box-shadow .2s; outline: none;
  -webkit-appearance: none;
}
.field input:focus, .field select:focus {
  border-color: var(--accent); box-shadow: 0 0 0 3px rgba(0,198,255,0.1);
}
.field input::placeholder { color: var(--muted); }

/* Password toggle */
.input-wrap { position: relative; }
.input-wrap input { padding-right: 48px; }
.eye-btn {
  position: absolute; right: 13px; top: 50%; transform: translateY(-50%);
  cursor: pointer; color: var(--muted); font-size: 1rem; user-select: none;
  background: none; border: none; line-height: 1;
}

/* ── BUTTONS ── */
.btn {
  display: inline-flex; align-items: center; justify-content: center; gap: 8px;
  padding: 13px 28px; border-radius: 10px; border: none; cursor: pointer;
  font-family: 'DM Sans', sans-serif; font-size: .95rem; font-weight: 500;
  transition: all .25s; text-decoration: none;
}
.btn-primary {
  background: linear-gradient(135deg, var(--accent2), var(--accent)); color: #fff;
}
.btn-primary:hover { transform: translateY(-2px); box-shadow: 0 10px 36px rgba(0,114,255,0.38); }
.btn-success {
  background: linear-gradient(135deg, #00a870, var(--success)); color: #fff;
}
.btn-success:hover { transform: translateY(-1px); box-shadow: 0 8px 24px rgba(0,229,160,0.3); }
.btn-outline {
  background: transparent; color: var(--text); border: 1px solid var(--border);
}
.btn-outline:hover { border-color: var(--accent); color: var(--accent); }
.btn-full { width: 100%; }
.btn-lg { padding: 15px 32px; font-size: 1.05rem; font-family: 'Syne', sans-serif; font-weight: 600; letter-spacing: .3px; }

/* ── SECTION CARD (inside dashboard/result) ── */
.section-card {
  background: var(--surface); border: 1px solid var(--border);
  border-radius: 20px; padding: 30px 32px; margin-bottom: 24px;
}
.section-title {
  font-family: 'Syne', sans-serif; font-weight: 700; font-size: 1.05rem;
  margin-bottom: 22px; display: flex; align-items: center; gap: 10px;
}
.icon-badge {
  width: 30px; height: 30px; border-radius: 8px; flex-shrink: 0;
  background: linear-gradient(135deg, var(--accent2), var(--accent));
  display: flex; align-items: center; justify-content: center; font-size: .85rem;
}

/* ── UPLOAD ZONE ── */
.upload-zone {
  border: 2px dashed var(--border); border-radius: 20px;
  padding: 52px 40px; text-align: center;
  background: var(--surface); transition: all .3s; cursor: pointer;
  position: relative; overflow: hidden; margin-bottom: 24px;
}
.upload-zone:hover, .upload-zone.drag-over {
  border-color: var(--accent);
  background: rgba(0,198,255,0.04);
  box-shadow: 0 0 0 4px rgba(0,198,255,0.06);
}
.upload-zone input[type="file"] {
  position: absolute; inset: 0; opacity: 0; cursor: pointer; z-index: 2;
  width: 100%; height: 100%;
}
.upload-icon { font-size: 3rem; margin-bottom: 14px; display: block; }
.upload-zone h3 { font-family: 'Syne', sans-serif; font-size: 1.15rem; margin-bottom: 8px; }
.upload-zone p  { color: var(--muted); font-size: .85rem; }
.preview-img {
  max-width: 100%; max-height: 260px; border-radius: 12px;
  margin-top: 22px; display: none; border: 1px solid var(--border); position: relative; z-index: 1;
}

/* ── ANALYZE BUTTON ── */
.analyze-btn {
  width: 100%; padding: 16px;
  background: linear-gradient(135deg, var(--accent2), var(--accent));
  color: #fff; border: none; border-radius: 12px;
  font-family: 'Syne', sans-serif; font-size: 1.05rem; font-weight: 700;
  cursor: pointer; transition: all .3s; letter-spacing: .4px;
  display: flex; align-items: center; justify-content: center; gap: 10px;
}
.analyze-btn:hover { transform: translateY(-2px); box-shadow: 0 12px 40px rgba(0,114,255,0.4); }

/* ── RESULT PAGE ── */
.result-grid { display: grid; grid-template-columns: 1fr 1.45fr; gap: 28px; align-items: start; }

/* Image card */
.img-card { background: var(--surface); border: 1px solid var(--border); border-radius: 20px; overflow: hidden; }
.img-card img { width: 100%; height: 280px; object-fit: cover; display: block; }
.img-card .img-meta { padding: 16px 20px; }
.img-card .img-meta h4 { font-size: .73rem; color: var(--muted); letter-spacing: .5px; text-transform: uppercase; margin-bottom: 4px; }
.img-card .img-meta p  { font-size: .9rem; font-weight: 500; }

/* Patient table card */
.patient-card { background: var(--surface); border: 1px solid var(--border); border-radius: 20px; padding: 26px 28px; margin-top: 22px; }
.info-table { width: 100%; border-collapse: collapse; font-size: .84rem; }
.info-table td { padding: 8px 0; border-bottom: 1px solid var(--border); vertical-align: top; }
.info-table tr:last-child td { border-bottom: none; }
.info-table td:first-child { color: var(--muted); width: 145px; }
.info-table td:last-child  { font-weight: 500; }

/* Result main card */
.result-card { background: var(--surface); border: 1px solid var(--border); border-radius: 20px; padding: 30px 32px; }
.result-label { font-size: .74rem; color: var(--muted); letter-spacing: .5px; text-transform: uppercase; margin-bottom: 12px; }

/* DR stage badge */
.dr-stage {
  display: inline-flex; align-items: center; gap: 10px;
  padding: 11px 22px; border-radius: 12px; margin-bottom: 10px;
  font-family: 'Syne', sans-serif; font-weight: 700; font-size: 1.2rem; border: 1px solid;
}
.stage-0 { background: rgba(0,229,160,0.1);  border-color: rgba(0,229,160,0.35);  color: #00e5a0; }
.stage-1 { background: rgba(255,224,100,0.1); border-color: rgba(255,224,100,0.35); color: #ffe064; }
.stage-2 { background: rgba(255,179,71,0.1);  border-color: rgba(255,179,71,0.35);  color: #ffb347; }
.stage-3 { background: rgba(255,120,50,0.1);  border-color: rgba(255,120,50,0.35);  color: #ff7832; }
.stage-4 { background: rgba(255,76,106,0.1);  border-color: rgba(255,76,106,0.35);  color: #ff4c6a; }

.stage-desc { color: var(--muted); font-size: .87rem; margin-bottom: 22px; line-height: 1.6; }

/* Confidence bar */
.conf-section { margin: 20px 0; }
.conf-row { display: flex; justify-content: space-between; margin-bottom: 8px; font-size: .85rem; }
.conf-pct { font-family: 'Syne', sans-serif; font-weight: 700; color: var(--accent); }
.bar-track { height: 10px; background: var(--surface2); border-radius: 999px; overflow: hidden; }
.bar-fill  { height: 100%; border-radius: 999px; background: linear-gradient(90deg, var(--accent2), var(--accent)); transition: width 1s cubic-bezier(.4,0,.2,1); }

/* Probability rows */
.probs-section { margin: 20px 0; }
.probs-label { font-size: .74rem; color: var(--muted); letter-spacing: .5px; text-transform: uppercase; margin-bottom: 12px; }
.prob-row { display: flex; align-items: center; gap: 12px; padding: 8px 0; border-bottom: 1px solid var(--border); font-size: .84rem; }
.prob-row:last-child { border-bottom: none; }
.prob-name { width: 150px; flex-shrink: 0; color: var(--muted); }
.prob-mini-track { flex: 1; height: 6px; background: var(--surface2); border-radius: 999px; overflow: hidden; }
.prob-mini-fill  { height: 100%; border-radius: 999px; transition: width .9s ease; }
.prob-pct { width: 48px; text-align: right; font-family: 'Syne', sans-serif; font-weight: 600; font-size: .82rem; }

/* Recommendation box */
.rec-box {
  margin-top: 18px; padding: 15px 18px; border-radius: 12px;
  border-left: 3px solid; font-size: .86rem; line-height: 1.65;
}
.rec-0 { background: rgba(0,229,160,0.07);  border-color: #00e5a0; }
.rec-1 { background: rgba(255,224,100,0.07); border-color: #ffe064; }
.rec-2 { background: rgba(255,179,71,0.07);  border-color: #ffb347; }
.rec-3 { background: rgba(255,120,50,0.07);  border-color: #ff7832; }
.rec-4 { background: rgba(255,76,106,0.07);  border-color: #ff4c6a; }

/* Result action buttons */
.result-actions { display: flex; gap: 12px; margin-top: 26px; }
.result-actions .btn { flex: 1; }

/* ── LOADING OVERLAY ── */
.loader-overlay {
  position: fixed; inset: 0; z-index: 200;
  background: rgba(3,11,20,0.9); backdrop-filter: blur(10px);
  display: none; align-items: center; justify-content: center;
  flex-direction: column; gap: 20px;
}
.loader-overlay.show { display: flex; }
.loader-ring {
  width: 76px; height: 76px; border-radius: 50%;
  border: 3px solid var(--border); border-top-color: var(--accent);
  animation: spin 1s linear infinite;
}
@keyframes spin { to { transform: rotate(360deg); } }
.loader-text { color: var(--muted); font-size: .9rem; }

/* ── TOAST ── */
.toast {
  position: fixed; bottom: 28px; right: 28px; z-index: 300;
  padding: 13px 20px; border-radius: 12px; font-size: .87rem;
  background: var(--surface2); border: 1px solid var(--border);
  transform: translateY(80px); opacity: 0;
  transition: all .35s cubic-bezier(.4,0,.2,1);
  max-width: 320px;
}
.toast.show   { transform: translateY(0); opacity: 1; }
.toast.success{ border-color: rgba(0,229,160,0.4); color: var(--success); }
.toast.error  { border-color: rgba(255,76,106,0.4); color: var(--danger); }

/* ── PAGE HEADER ROW ── */
.page-header { display: flex; align-items: flex-start; justify-content: space-between; margin-bottom: 30px; flex-wrap: wrap; gap: 12px; }
.page-header h2 { font-family: 'Syne', sans-serif; font-size: 1.75rem; font-weight: 700; }
.page-header .meta { color: var(--muted); font-size: .87rem; margin-top: 4px; }

/* ── ANIMATIONS ── */
@keyframes fadeUp {
  from { opacity: 0; transform: translateY(22px); }
  to   { opacity: 1; transform: translateY(0); }
}
.fade-up { animation: fadeUp .5s ease both; }
.fade-up-1 { animation: fadeUp .5s .1s ease both; }
.fade-up-2 { animation: fadeUp .5s .2s ease both; }
.fade-up-3 { animation: fadeUp .5s .3s ease both; }

/* ── RESPONSIVE ── */
@media (max-width: 900px) {
  .result-grid { grid-template-columns: 1fr; }
}
@media (max-width: 768px) {
  nav { padding: 16px 20px; }
  .page-body { padding: 80px 16px 40px; }
  .form-grid { grid-template-columns: 1fr; }
  .form-row  { flex-direction: column; }
  .glass-card { padding: 30px 22px; }
  .upload-zone { padding: 40px 20px; }
}


Overwriting static/style.css


In [103]:
%%writefile fix_db.py

# ============================================================
#  RUN THIS ONCE to fix the database schema error
#  Error: "table users has 2 columns but 4 values were supplied"
#  Cause: Old database.db still has the 2-column schema
#  Fix:   Drop the old table and recreate with 4 columns
# ============================================================

import sqlite3
import os

DB_PATH = "database.db"

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# Check current schema
cur.execute("PRAGMA table_info(users)")
cols = cur.fetchall()
print("Current columns:", [c[1] for c in cols])

# Drop old table
cur.execute("DROP TABLE IF EXISTS users")
print("Old table dropped.")

# Recreate with correct 4-column schema
cur.execute("""
    CREATE TABLE users (
        username  TEXT PRIMARY KEY,
        password  TEXT NOT NULL,
        full_name TEXT DEFAULT '',
        hospital  TEXT DEFAULT ''
    )
""")
conn.commit()
conn.close()
print("New table created with 4 columns: username, password, full_name, hospital")
print("Done! Now restart your Flask app and register again.")

Overwriting fix_db.py


In [104]:
# In a new cell:
!python fix_db.py

Current columns: ['username', 'password', 'full_name', 'hospital']
Old table dropped.
New table created with 4 columns: username, password, full_name, hospital
Done! Now restart your Flask app and register again.


In [105]:
# ============================
# ✅ Run Flask in background
# ============================
!pkill -f flask


In [108]:
!lsof -i :8000

In [109]:
!kill -9 11832

/bin/bash: line 1: kill: (11832) - No such process


In [110]:
!nohup python app.py > flask.log 2>&1 &

In [111]:
!pip install pyngrok

In [112]:
!tail -n 50 flask.log

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
Press CTRL+C to quit


In [113]:
# ============================
# ✅ Start ngrok tunnel
# ============================
from pyngrok import ngrok, conf

conf.get_default().auth_token = "3AhnJqh95MllATs6bXVcbCKMxwS_34aAyYotQEKkvR8mZw7PJ"   # 🔑 put your token here
public_url = ngrok.connect(8000)
print("🌍 Public URL:", public_url)

🌍 Public URL: NgrokTunnel: "https://unpelted-denny-unhedonistic.ngrok-free.dev" -> "http://localhost:8000"
